In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

# Load the dataset from the local JSON file
raw_dataset = load_dataset('json', data_files='instructions_dataset.json')

# Split the 'train' split into 90% training and 10% validation
dataset_splits = raw_dataset['train'].train_test_split(test_size=0.1, seed=42)

print(f"Dataset prepared: {dataset_splits}")

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = 'google/flan-t5-base'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Detect device and move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Model and tokenizer loaded successfully.")
print(f"Model is currently on device: {model.device}")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print("LoRA configuration applied successfully.")

In [ ]:
def preprocess_function(examples):
    # Tokenize both inputs and targets in a single call using text_target
    model_inputs = tokenizer(
        examples['instruction'],
        text_target=examples['output'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    return model_inputs

# Apply the fixed function to the dataset and remove original columns
tokenized_dataset = dataset_splits.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_splits['train'].column_names
)

print(f"Tokenized dataset columns: {tokenized_dataset['train'].column_names}")
print("Sample labels check:", 'labels' in tokenized_dataset['train'].column_names)

In [ ]:
from transformers import Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Initialize the data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=tokenizer.pad_token_id
)

# Define the training arguments optimized for T4 GPU
# Updated 'evaluation_strategy' to 'eval_strategy' to support newer transformers versions
training_args = Seq2SeqTrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_steps=10,
    push_to_hub=False,
    report_to='none'
)

print('Training arguments and data collator configured.')

In [ ]:
from transformers import Seq2SeqTrainer

# Initialize the Seq2SeqTrainer
# Note: Removed the 'tokenizer' argument as it is already handled by the data_collator
# and can cause a TypeError in certain versions of the Trainer class.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator
)

print("Starting training...")
# Execute the fine-tuning process
trainer.train()

print("Training completed.")

In [ ]:
test_instructions = [
    'Explain Docker',
    'What is a CI/CD pipeline?',
    'What is the purpose of Kubernetes?',
    'Define Terraform and its use case.'
]

print('--- Model Inference Results ---')
model.eval()

for instruction in test_instructions:
    # Tokenize input
    inputs = tokenizer(instruction, return_tensors='pt').to(device)

    # Generate output tokens
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            max_new_tokens=100,
            do_sample=True,
            top_p=0.9
        )

    # Decode output
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    print(f'Instruction: {instruction}')
    print(f'Response: {response}')
    print('-' * 30)

In [ ]:
import json

# Defining a robust dataset with 50+ unique technical definitions
# Fixed the missing quotes around the PaaS output value
tech_data = [
    {'instruction': 'What is Docker?', 'output': 'Docker is an open-source platform that automates the deployment, scaling, and management of applications using containerization.'},
    {'instruction': 'Explain CI/CD.', 'output': 'CI/CD stands for Continuous Integration and Continuous Deployment/Delivery, a method to frequently deliver apps to customers by introducing automation into the stages of app development.'},
    {'instruction': 'Define Terraform.', 'output': 'Terraform is an open-source infrastructure as code (IaC) software tool that allows users to define and provide data center infrastructure using a declarative configuration language.'},
    {'instruction': 'What is Kubernetes?', 'output': 'Kubernetes (K8s) is an open-source system for automating deployment, scaling, and management of containerized applications.'},
    {'instruction': 'What is AWS Lambda?', 'output': 'AWS Lambda is a serverless computing service that lets you run code without provisioning or managing servers, automatically scaling with the size of the workload.'},
    {'instruction': 'Define Ansible.', 'output': 'Ansible is an open-source IT automation engine that automates provisioning, configuration management, application deployment, and intra-service orchestration.'},
    {'instruction': 'What is a Microservices architecture?', 'output': 'Microservices architecture is a design approach to build a single application as a suite of small services, each running in its own process and communicating with lightweight mechanisms.'},
    {'instruction': 'Explain Git.', 'output': 'Git is a distributed version control system that tracks changes in any set of computer files, usually used for coordinating work among programmers collaborating on source code.'},
    {'instruction': 'What is Jenkins?', 'output': 'Jenkins is an open-source automation server that helps automate the parts of software development related to building, testing, and deploying, facilitating CI/CD.'},
    {'instruction': 'Define Serverless Computing.', 'output': 'Serverless computing is a cloud execution model where the cloud provider runs the server and dynamically manages the allocation of machine resources.'},
    {'instruction': 'What is Prometheus?', 'output': 'Prometheus is an open-source systems monitoring and alerting toolkit originally built at SoundCloud.'},
    {'instruction': 'Explain Grafana.', 'output': 'Grafana is a multi-platform open-source analytics and interactive visualization web application that provides charts, graphs, and alerts for supported data sources.'},
    {'instruction': 'What is an API Gateway?', 'output': 'An API Gateway is a management tool that sits between a client and a collection of backend services, acting as a single entry point for API requests.'},
    {'instruction': 'Define Redis.', 'output': 'Redis is an open-source, in-memory data structure store, used as a distributed, in-memory key–value database, cache, and message broker.'},
    {'instruction': 'What is SQL?', 'output': 'SQL (Structured Query Language) is a standard language for managing data held in a relational database management system.'},
    {'instruction': 'What is NoSQL?', 'output': 'NoSQL is a non-tabular database that stores data differently than relational tables, such as document-based, key-value, or graph formats.'},
    {'instruction': 'Define Python.', 'output': 'Python is a high-level, interpreted programming language known for its readability and versatile use in web development, data science, and automation.'},
    {'instruction': 'What is a Load Balancer?', 'output': 'A load balancer is a device or software that distributes network or application traffic across a cluster of servers to ensure high availability and reliability.'},
    {'instruction': 'Explain Virtualization.', 'output': 'Virtualization is the process of creating a virtual version of something, such as hardware platforms, storage devices, and network resources.'},
    {'instruction': 'What is a Firewall?', 'output': 'A firewall is a network security system that monitors and controls incoming and outgoing network traffic based on predetermined security rules.'},
    {'instruction': 'Define DNS.', 'output': 'DNS (Domain Name System) is the hierarchical and decentralized naming system for computers, services, or other resources connected to the Internet.'},
    {'instruction': 'What is HTTP/HTTPS?', 'output': 'HTTP is the protocol for transferring data over the web, while HTTPS is the secure version using SSL/TLS encryption.'},
    {'instruction': 'Explain TCP/IP.', 'output': 'TCP/IP is a suite of communication protocols used to interconnect network devices on the internet.'},
    {'instruction': 'What is a VPN?', 'output': 'A VPN (Virtual Private Network) creates a private network from a public internet connection, providing online privacy and anonymity.'},
    {'instruction': 'Define SSL/TLS.', 'output': 'SSL (Secure Sockets Layer) and its successor TLS (Transport Layer Security) are protocols for establishing authenticated and encrypted links between networked computers.'},
    {'instruction': 'What is Bash?', 'output': 'Bash is a Unix shell and command language used as the default login shell for most Linux distributions.'},
    {'instruction': 'Explain Agile methodology.', 'output': 'Agile is an iterative approach to project management and software development that helps teams deliver value to customers faster and with fewer headaches.'},
    {'instruction': 'What is Scrum?', 'output': 'Scrum is a framework within which people can address complex adaptive problems, while productively and creatively delivering products of the highest possible value.'},
    {'instruction': 'Define Kanban.', 'output': 'Kanban is a visual system for managing work as it moves through a process, focusing on just-in-time delivery while not overloading team members.'},
    {'instruction': 'What is Cloud Computing?', 'output': 'Cloud computing is the on-demand availability of computer system resources, especially data storage and computing power, without direct active management by the user.'},
    {'instruction': 'Explain SaaS.', 'output': 'SaaS (Software as a Service) is a software licensing and delivery model in which software is licensed on a subscription basis and is centrally hosted.'},
    {'instruction': 'What is PaaS?', 'output': 'PaaS (Platform as a Service) is a category of cloud computing services that provides a platform allowing customers to develop, run, and manage applications.'},
    {'instruction': 'What is IaaS?', 'output': 'IaaS (Infrastructure as a Service) is a form of cloud computing that provides virtualized computing resources over the internet.'},
    {'instruction': 'Define Public Cloud.', 'output': 'A public cloud is a type of computing where a service provider makes resources available to the general public over the internet.'},
    {'instruction': 'What is a Private Cloud?', 'output': 'A private cloud consists of computing resources used exclusively by one business or organization.'},
    {'instruction': 'Explain Hybrid Cloud.', 'output': 'Hybrid cloud is a computing environment that combines a public cloud and a private cloud by allowing data and applications to be shared between them.'},
    {'instruction': 'What is Data Science?', 'output': 'Data science is an interdisciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge and insights from structured and unstructured data.'},
    {'instruction': 'Define Machine Learning.', 'output': 'Machine learning is a branch of AI and computer science which focuses on the use of data and algorithms to imitate the way that humans learn, gradually improving its accuracy.'},
    {'instruction': 'What is Artificial Intelligence?', 'output': 'Artificial intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems.'},
    {'instruction': 'Explain Deep Learning.', 'output': 'Deep learning is a subset of machine learning based on artificial neural networks with representation learning.'},
    {'instruction': 'What is an IDE?', 'output': 'An IDE (Integrated Development Environment) is a software application that provides comprehensive facilities to computer programmers for software development.'},
    {'instruction': 'Define JSON.', 'output': 'JSON (JavaScript Object Notation) is a lightweight data-interchange format that is easy for humans to read and write and easy for machines to parse and generate.'},
    {'instruction': 'What is XML?', 'output': 'XML (eXtensible Markup Language) is a markup language that defines a set of rules for encoding documents in a format that is both human-readable and machine-readable.'},
    {'instruction': 'Explain REST API.', 'output': 'A REST API is an application programming interface that conforms to the constraints of REST architectural style and allows for interaction with RESTful web services.'},
    {'instruction': 'What is GraphQL?', 'output': 'GraphQL is a query language for APIs and a runtime for fulfilling those queries with your existing data.'},
    {'instruction': 'Define Unit Testing.', 'output': 'Unit testing is a software testing method by which individual units of source code are tested to determine whether they are fit for use.'},
    {'instruction': 'What is Integration Testing?', 'output': 'Integration testing is a phase in software testing in which individual software modules are combined and tested as a group.'},
    {'instruction': 'Explain Regression Testing.', 'output': 'Regression testing is re-running functional and non-functional tests to ensure that previously developed and tested software still performs after a change.'},
    {'instruction': 'What is Penetration Testing?', 'output': 'A penetration test (pen test) is an authorized simulated cyberattack on a computer system, performed to evaluate the security of the system.'},
    {'instruction': 'Define Encryption.', 'output': 'Encryption is the process of converting information or data into a code, especially to prevent unauthorized access.'},
    {'instruction': 'What is Hashing?', 'output': 'Hashing is the process of transforming a given string of characters into another value, usually representing the original string by a shorter, fixed-length value or key.'},
    {'instruction': 'Explain OAuth 2.0.', 'output': 'OAuth 2.0 is an industry-standard protocol for authorization, allowing a website or application to access resources hosted by other web apps on behalf of a user.'}
]

# Writing to instructions_dataset.json
with open('instructions_dataset.json', 'w') as f:
    json.dump(tech_data, f, indent=4)

# Verification
with open('instructions_dataset.json', 'r') as f:
    loaded_data = json.load(f)

print(f'Dataset saved successfully to instructions_dataset.json.')
print(f'Verified total number of entries: {len(loaded_data)}')

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

model_name = 'google/flan-t5-base'

# Re-load tokenizer and model to clear any previous state
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move fresh model to T4 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define new LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Apply LoRA adapters to the newly loaded model
model = get_peft_model(model, lora_config)

# Print trainable parameters to verify setup
model.print_trainable_parameters()
print(f"Model re-initialized and moved to: {device}")

In [ ]:
from datasets import load_dataset

# Load the expanded dataset
raw_dataset = load_dataset('json', data_files='instructions_dataset.json')

# Split into 90% training and 10% validation
dataset_splits = raw_dataset['train'].train_test_split(test_size=0.1, seed=42)

def preprocess_function(examples):
    # Tokenize inputs and labels using text_target
    return tokenizer(
        examples['instruction'],
        text_target=examples['output'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )

# Apply preprocessing and remove original text columns
tokenized_dataset = dataset_splits.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_splits['train'].column_names
)

print(f"Dataset splits: {tokenized_dataset}")
print(f"Columns in tokenized dataset: {tokenized_dataset['train'].column_names}")

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Initialize the data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=tokenizer.pad_token_id
)

# Define the training arguments optimized for stable convergence on T4 GPU
training_args = Seq2SeqTrainingArguments(
    output_dir='./results_refined',
    eval_strategy='epoch',
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
    logging_steps=5,
    push_to_hub=False,
    report_to='none'
)

# Initialize the Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator
)

print("Starting refined fine-tuning...")
# Execute the fine-tuning process
trainer.train()

print("Fine-tuning completed successfully.")

In [ ]:
# Debug: Check if labels are correctly tokenized
example_idx = 0
input_ids = tokenized_dataset['train'][example_idx]['input_ids']
labels = tokenized_dataset['train'][example_idx]['labels']
print(f'Input: {tokenizer.decode(input_ids, skip_special_tokens=True)}')
print(f'Label: {tokenizer.decode([l if l != -100 else tokenizer.pad_token_id for l in labels], skip_special_tokens=True)}')

# Re-initialize training with a more conservative learning rate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

refined_training_args = Seq2SeqTrainingArguments(
    output_dir='./results_final_attempt',
    eval_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=True,
    logging_steps=5,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=refined_training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator
)

print('Starting training with lower learning rate and more epochs...')
trainer.train()
print('Training finished.')

In [ ]:
test_queries = [
    'What is Docker?',
    'Explain CI/CD.',
    'Define Terraform.'
]

print('--- Final Refined Model Inference Results ---')
model.eval()

for instruction in test_queries:
    # Tokenize input and move to GPU
    inputs = tokenizer(instruction, return_tensors='pt').to(device)

    # Generate output tokens (deterministic for validation)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            max_new_tokens=100,
            do_sample=False
        )

    # Decode output
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    print(f'Instruction: {instruction}')
    print(f'Response: {response}')
    print('-' * 40)

In [ ]:
print('--- Decoded Training Samples ---')
for i in range(3):
    sample = tokenized_dataset['train'][i]
    input_ids = sample['input_ids']
    label_ids = sample['labels']

    # Convert -100 back to pad token ID for decoding
    clean_labels = [token if token != -100 else tokenizer.pad_token_id for token in label_ids]

    input_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    label_text = tokenizer.decode(clean_labels, skip_special_tokens=True)

    print(f'Example {i+1}:')
    print(f'Decoded Input: {input_text}')
    print(f'Decoded Label: {label_text}')
    print('-' * 20)

In [ ]:
print('--- Verifying Training Data Decoding ---')
for i in range(3):
    input_ids = tokenized_dataset['train'][i]['input_ids']
    label_ids = tokenized_dataset['train'][i]['labels']

    # Replace -100 with pad token for decoding
    clean_labels = [token if token != -100 else tokenizer.pad_token_id for token in label_ids]

    print(f'Example {i+1}:')
    print(f'Decoded Input: {tokenizer.decode(input_ids, skip_special_tokens=True)}')
    print(f'Decoded Label: {tokenizer.decode(clean_labels, skip_special_tokens=True)}')

In [ ]:
# Display the first 5 examples from the tokenized training split
import pandas as pd

train_df = pd.DataFrame(tokenized_dataset['train'].select(range(5)))
display(train_df)

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

model_name = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Increase Rank (r) and Alpha for stronger adaptation on small dataset
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Using a significantly higher learning rate for the small dataset to force updates
aggressive_args = Seq2SeqTrainingArguments(
    output_dir='./results_aggressive',
    eval_strategy='no',
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    num_train_epochs=20,
    weight_decay=0.01,
    fp16=True,
    logging_steps=5,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=aggressive_args,
    train_dataset=tokenized_dataset['train'],
    data_collator=data_collator
)

print("Starting aggressive fine-tuning...")
trainer.train()
print("Training complete.")

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

# 1. & 2. Load fresh model and tokenizer, move to T4 GPU
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Define a higher rank LoraConfig (r=16)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# 4. Apply LoRA configuration to the model
model = get_peft_model(model, lora_config)

# 5. Initialize aggressive Seq2SeqTrainingArguments
aggressive_args = Seq2SeqTrainingArguments(
    output_dir="./results_aggressive",
    eval_strategy="no",
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    num_train_epochs=20,
    weight_decay=0.01,
    fp16=True,
    logging_steps=5,
    report_to="none"
)

# 6. Instantiate the Seq2SeqTrainer
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=aggressive_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

# 7. Execute training
print("Starting aggressive fine-tuning (r=16, LR=1e-3, Epochs=20)...")
trainer.train()
print("Aggressive training completed.")

In [ ]:
print('--- Decoded Training Samples ---')
for i in range(3):
    sample = tokenized_dataset['train'][i]
    input_ids = sample['input_ids']
    label_ids = sample['labels']

    # Convert -100 back to pad token ID for decoding
    clean_labels = [token if token != -100 else tokenizer.pad_token_id for token in label_ids]

    input_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    label_text = tokenizer.decode(clean_labels, skip_special_tokens=True)

    print(f'Example {i+1}:')
    print(f'Decoded Input: {input_text}')
    print(f'Decoded Label: {label_text}')
    print('-' * 20)